# move-iit-hpe — Energy and real-time factor

**Protocol aligned with `ExpEnergy/measure_energy.py` and `ExpFinal/final_analysis.py`.**

Inputs: completed host-side energy session (12 sequences × 3 configurations × 5 repetitions). The measurement script and the final analyzer are separate; **this notebook never reruns inference**. Execute the analyzer with `--strict` before interpreting plots.

**Definitions:** CPU = package-level Intel RAPL, GPU = whole-GPU NVML; combined energy is their sum. Runtime includes startup, processing and shutdown; excludes cooldown. $\mathrm{RTF}=\sum_s\overline T_s/\sum_s D_s$ where $D_s$ is the input span derived from the offlineLoader timestamp-range log, not output CSV duration. For energy the dataset mean is the equally-weighted mean of sequence-level five-repeat means. CI: 95% percentile bootstrap, 5,000 resamples of the 12 sequence units, fixed seed `20260911`.

**Comparability caveat:** the older eH36M and DHP19 measurements remain unmodified. Their archived settings (especially DHP19 ROI) must be audited before joining any new accuracy trade-off plots.

## 1. Setup: session path and shared figure style

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display

# Edit this directory if your energy result root differs (HOST by default).
SESSION_DIR = Path(os.environ.get(
    'MOVEIIT_ENERGY_SESSION_DIR',
    str(Path.home() / 'data/MoveEnet_OFK_results/Energy/moveiit_energy_rtf_roi35_gain1_v2')
))
FIG_DIR = SESSION_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_FORMATS = ('svg', 'pdf', 'png')
SAVE_FIGURES = True
FIGSIZE = (8.2, 4.7)  # match ExpFinal plot_runtime / plot_energy figures
FONTSIZE = 11
TITLE_SIZE = 13
DPI = 300

# One style block for all figures; SVG labels stay editable.
COLOR = {'movenet_50hz': '#45678A', 'movenet_5hz': '#98AEC5',
         'moveenetofk_5hz': '#D17E46'}
CPU_COLOR, GPU_COLOR = '#45678A', '#D17E46'
PLOT_ORDER = ['movenet_50hz', 'movenet_5hz', 'moveenetofk_5hz']
LABEL = {'movenet_50hz': 'MoveEnet only (50 Hz)',
         'movenet_5hz': 'MoveEnet only (5 Hz)',
         'moveenetofk_5hz': 'MoveEnetOFK (5 Hz / OF 200 Hz)'}

mpl.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': FONTSIZE,
    'axes.titlesize': TITLE_SIZE, 'axes.labelsize': FONTSIZE,
    'xtick.labelsize': FONTSIZE - 1, 'ytick.labelsize': FONTSIZE - 1,
    'legend.fontsize': FONTSIZE - 1, 'svg.fonttype': 'none',
    'axes.spines.top': False, 'axes.spines.right': False,
    'savefig.facecolor': 'white', 'figure.facecolor': 'white',
})
print('Session:', SESSION_DIR)
print('Figures:', FIG_DIR)


## 2. Load validated analysis tables

In [ ]:
required = {
    'summary': SESSION_DIR / 'energy_runtime_dataset_summary.csv',
    'samples': SESSION_DIR / 'energy_runtime_by_sample.csv',
    'pairs': SESSION_DIR / 'paired_differences.csv',
}
missing = [str(p) for p in required.values() if not p.is_file()]
if missing:
    raise FileNotFoundError('Run analyze_moveiit_energy_rtf.py --strict first. Missing: ' + ', '.join(missing))
summary = pd.read_csv(required['summary']).set_index('model').loc[PLOT_ORDER].reset_index()
seq = pd.read_csv(required['samples'])
pairs = pd.read_csv(required['pairs'])
assert summary['n_sequences'].eq(12).all(), 'Incomplete final 12-sequence analysis'
assert summary['repetitions_per_sequence'].eq(5).all(), 'Expected five repetitions'
assert len(seq)==36 and len(pairs)==2, 'Unexpected number of method/sequence comparisons'
assert np.isfinite(summary.select_dtypes(include='number')).all().all(), 'Nonfinite summary values'
print('12 recordings; 5 repeats; 3 operating points; offlineLoader event duration')
display(summary[['method', 'energy_mean_j', 'cpu_energy_mean_j', 'gpu_energy_mean_j',
                 'wall_mean_s', 'real_time_factor', 'energy_j_per_input_second',
                 'energy_ci95_low', 'energy_ci95_high',
                 'rtf_ci95_low', 'rtf_ci95_high']].round(3))
display(pairs.round(3))


## 3. Plot helpers

In [ ]:
def save_figure(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        for ext in EXPORT_FORMATS:
            fig.savefig(FIG_DIR / f'{name}.{ext}', dpi=DPI,
                        bbox_inches='tight')
    plt.show()

def asym_ci(values, lows, highs):
    values = np.asarray(values, dtype=float)
    lows = np.asarray(lows, dtype=float)
    highs = np.asarray(highs, dtype=float)
    return np.vstack([np.maximum(0., values - lows), np.maximum(0., highs - values)])

x = np.arange(len(PLOT_ORDER))
labels = [LABEL[m] for m in PLOT_ORDER]


## 4. Total measured energy — CPU + GPU (one cell = one figure)

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.bar(x, summary['cpu_energy_mean_j'], color=CPU_COLOR, label='CPU package')
ax.bar(x, summary['gpu_energy_mean_j'], bottom=summary['cpu_energy_mean_j'],
       color=GPU_COLOR, label='GPU device')
ax.set_xticks(x, labels, rotation=15, ha='right')
ax.set_ylabel('Mean measured energy (J / sequence)')
ax.set_title('move-iit-hpe | CPU + GPU energy')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
save_figure(fig, 'moveiit_energy_components')


## 5. Energy (total) with sequence-bootstrap confidence intervals

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
v = summary['energy_mean_j'].to_numpy()
e = asym_ci(v, summary['energy_ci95_low'], summary['energy_ci95_high'])
ax.bar(x, v, color=[COLOR[m] for m in PLOT_ORDER], yerr=e, capsize=4,
       error_kw={'elinewidth': 1.2})
ax.set_xticks(x, labels, rotation=15, ha='right')
ax.set_ylabel('Mean measured energy (J / sequence)')
ax.set_title('move-iit-hpe | Total measured energy (95% bootstrap CI)')
ax.grid(axis='y', alpha=.2)
ax.set_axisbelow(True)
save_figure(fig, 'moveiit_energy_total_ci95')


## 6. Real-time factor (whole-command runtime / input-event span)

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
v = summary['real_time_factor'].to_numpy()
e = asym_ci(v, summary['rtf_ci95_low'], summary['rtf_ci95_high'])
ax.bar(x, v, color=[COLOR[m] for m in PLOT_ORDER], yerr=e, capsize=4,
       error_kw={'elinewidth': 1.2})
ax.axhline(1, color='#444444', linestyle='--', linewidth=1.2, label='RTF = 1')
ax.set_xticks(x, labels, rotation=15, ha='right')
ax.set_ylabel('Real-time factor (whole-command time / input span)')
ax.set_title('move-iit-hpe | Offline replay RTF (95% bootstrap CI)')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=.2)
ax.set_axisbelow(True)
save_figure(fig, 'moveiit_runtime_rtf')


## 7. Whole-sequence processing time

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
v = summary['wall_mean_s'].to_numpy()
e = asym_ci(v, summary['wall_ci95_low'], summary['wall_ci95_high'])
ax.bar(x, v, color=[COLOR[m] for m in PLOT_ORDER], yerr=e, capsize=4,
       error_kw={'elinewidth': 1.2})
ax.set_xticks(x, labels, rotation=15, ha='right')
ax.set_ylabel('Whole-sequence wall-clock time (s)')
ax.set_title('move-iit-hpe | Existing energy-run timing (95% bootstrap CI)')
ax.grid(axis='y', alpha=.2)
ax.set_axisbelow(True)
save_figure(fig, 'moveiit_runtime_wall_seconds')


## 8. Input-time-normalized energy (optional)

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
v = summary['energy_j_per_input_second'].to_numpy()
e = asym_ci(v, summary['normalized_energy_ci95_low'],
            summary['normalized_energy_ci95_high'])
ax.bar(x, v, color=[COLOR[m] for m in PLOT_ORDER], yerr=e, capsize=4,
       error_kw={'elinewidth': 1.2})
ax.set_xticks(x, labels, rotation=15, ha='right')
ax.set_ylabel('Measured energy per second of input (J / input-s)')
ax.set_title('move-iit-hpe | Input-normalized energy (95% bootstrap CI)')
ax.grid(axis='y', alpha=.2)
ax.set_axisbelow(True)
save_figure(fig, 'moveiit_normalized_energy')


## 9. Sample-level audit (optional)

In [ ]:
audit = seq.pivot(index=['sample_id', 'subject', 'sequence'], columns='model',
                  values=['energy_j', 'real_time_factor'])
display(audit.round(3))
print('Method comparison: baseline 50 Hz and baseline 5 Hz against OFK 5 Hz')
display(pairs[['reference', 'energy_saving_pct', 'energy_saving_ci95_low',
               'energy_saving_ci95_high', 'runtime_saving_pct',
               'runtime_saving_ci95_low', 'runtime_saving_ci95_high']].round(3))


## Interpretation and limits

This notebook visualizes **measured** CPU-package plus GPU-device energy, not process-isolated power. The five measurement repeats control noise; uncertainty is obtained by **resampling the 12 recordings**, not pretending all 60 repeats are independent. The RTF includes initialization/shutdown and measures **offline replay throughput**, not per-update inference latency. For paper comparisons with existing eH36M and DHP19 data, compare matching measurement definitions and disclose differing input datasets, network rates, checkpoints, hardware sessions and ROI settings where applicable.